In [1]:
import pandas as pd
import numpy as np
import os
import datetime

# Preprocress GHCN raw

In [7]:
folder = '/home/camarada/Documents/projects/temp-grss-nasa/ghcn_noaa/data/ghcnm.v4.0.1.20260330'
inv_file = "ghcnm.tavg.v4.0.1.20260330.qcf.inv"
dat_file = "ghcnm.tavg.v4.0.1.20260330.qcf.dat"

In [8]:
# Format: ID(11), LAT(9), LON(10), ELEV(7), NAME(30)
inv_specs = [(0, 11), (12, 20), (21, 30), (31, 37), (38, 68)]
inv_names = ['ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME']

df_inv = pd.read_fwf(os.path.join(folder,inv_file), colspecs=inv_specs, names=inv_names)

In [9]:
df_inv.head()

,ID,LATITUDE,LONGITUDE,ELEVATION,NAME
0,ACW00011604,57.7667,11.8667,18.0,SAVE
1,AE000041196,25.3330,55.5170,34.0,SHARJAH_INTER_AIRP
2,AEM00041184,25.6170,55.9330,31.0,RAS_AL_KHAIMAH_INTE
3,AEM00041194,25.2550,55.3640,10.4,DUBAI_INTL
4,AEM00041216,24.4300,54.4700,3.0,ABU_DHABI_BATEEN_AIR


In [12]:
# --- 3. PARSE DATA (.dat) ---
# Structure: ID(11), Year(4), Element(4), then 12 months of [Value(5), DM(1), QC(1), DS(1)]
# We define the starting columns:
data_specs = [(0, 11), (11, 15), (15, 19)]
data_names = ['ID', 'YEAR', 'ELEMENT']

for i in range(1, 13):
    start = 19 + (i - 1) * 8
    data_specs.append((start, start + 5))      # Value
    data_specs.append((start + 5, start + 6))  # DM Flag
    data_specs.append((start + 6, start + 7))  # QC Flag
    data_specs.append((start + 7, start + 8))  # DS Flag
    
    data_names.extend([f'VALUE_{i}', f'DM_{i}', f'QC_{i}', f'DS_{i}'])

df_data = pd.read_fwf(os.path.join(folder,dat_file), colspecs=data_specs, names=data_names)

In [20]:
# --- 4. CLEANING & POST-PROCESSING ---
# Convert -9999 to NaN and scale temperatures (stored as 100 * Celsius)
value_cols = [f'VALUE_{i}' for i in range(1, 13)]
for col in value_cols:
    df_data[col] = df_data[col].replace(-9999, np.nan) / 100.0

# --- 5. MERGE & SAVE TO PARQUET ---
# Merge inventory info (Lat/Lon/Name) into the data dataframe
df_final = pd.merge(df_data, df_inv, on='ID', how='left')

# Save to Parquet (requires 'pyarrow' or 'fastparquet' installed)
df_final.to_parquet(f"data/ghcn_monthly_v4-{datetime.datetime.now().strftime("%Y-%m-%d-%H-%m")}.parquet", compression='snappy', index=False)

print(f"converted {len(df_final)} rows to Parquet.")
print(df_final.head())

Successfully converted 1489877 rows to Parquet.
            ID  YEAR ELEMENT  VALUE_1  DM_1 QC_1 DS_1  VALUE_2  DM_2 QC_2  \
0  ACW00011604  1961    TAVG  -0.0085   NaN  NaN    k   0.0240   NaN  NaN   
1  ACW00011604  1962    TAVG   0.0117   NaN  NaN    k   0.0089   NaN  NaN   
2  ACW00011604  1963    TAVG  -0.0709   NaN  NaN    k  -0.0549   NaN  NaN   
3  ACW00011604  1964    TAVG   0.0066   NaN  NaN    k  -0.0081   NaN  NaN   
4  ACW00011604  1965    TAVG   0.0048   NaN  NaN    k  -0.0101   NaN  NaN   

   ... QC_11  DS_11  VALUE_12 DM_12 QC_12  DS_12  LATITUDE LONGITUDE  \
0  ...   NaN      k   -0.0035   NaN   NaN      k   57.7667   11.8667   
1  ...   NaN      k   -0.0122   NaN   NaN      k   57.7667   11.8667   
2  ...   NaN      k   -0.0104   NaN   NaN      k   57.7667   11.8667   
3  ...   NaN      k    0.0116   NaN   NaN      k   57.7667   11.8667   
4  ...   NaN      k   -0.0174   NaN   NaN      k   57.7667   11.8667   

  ELEVATION  NAME  
0      18.0  SAVE  
1      18.0  SAV

## Load parquet back

In [2]:
df = pd.read_parquet('/home/camarada/Documents/projects/temp-grss-nasa/ghcn_noaa/data/ghcn_monthly_v4-2026-03-31-19-03.parquet')

In [6]:
#3 remove df object to release memory
del df

In [3]:
# we filter the data for a period higher than 1980
df_final = df.loc[df['YEAR'] >= 1990]
print(df_final.shape)

(560021, 55)


In [4]:
df_final.columns

Index(['ID', 'YEAR', 'ELEMENT', 'VALUE_1', 'DM_1', 'QC_1', 'DS_1', 'VALUE_2',
       'DM_2', 'QC_2', 'DS_2', 'VALUE_3', 'DM_3', 'QC_3', 'DS_3', 'VALUE_4',
       'DM_4', 'QC_4', 'DS_4', 'VALUE_5', 'DM_5', 'QC_5', 'DS_5', 'VALUE_6',
       'DM_6', 'QC_6', 'DS_6', 'VALUE_7', 'DM_7', 'QC_7', 'DS_7', 'VALUE_8',
       'DM_8', 'QC_8', 'DS_8', 'VALUE_9', 'DM_9', 'QC_9', 'DS_9', 'VALUE_10',
       'DM_10', 'QC_10', 'DS_10', 'VALUE_11', 'DM_11', 'QC_11', 'DS_11',
       'VALUE_12', 'DM_12', 'QC_12', 'DS_12', 'LATITUDE', 'LONGITUDE',
       'ELEVATION', 'NAME'],
      dtype='object')

In [7]:
## convert the data to a long format


# 2. Use wide_to_long to transform the dataframe
# stubnames: the prefixes of the columns we want to stack (VALUE and QC)
# i: the columns to keep as unique identifiers
# j: the name of the new column that will hold the month number (1-12)
df_long = pd.wide_to_long(
    df_final, 
    stubnames=['VALUE', 'QC', 'DM', 'DS'], 
    i=['ID', 'YEAR', 'LATITUDE', 'LONGITUDE', 'NAME'], 
    j='MONTH', 
    sep='_'
).reset_index()

# 3. Final Polish
# Rename 'VALUE' to 'TEMPERATURE' for clarity
df_long = df_long.rename(columns={'VALUE': 'TEMPERATURE', 'QC': 'QC_FLAG'})

# Sort by station and date
df_long = df_long.sort_values(['ID', 'YEAR', 'MONTH'])


In [11]:
df_long.to_parquet('data/long_format_ghcn1990-2026.parquet', compression='snappy', index=False)

In [16]:
df_long['DS'].unique()

array(['{', '}', None, '|', 'k', 'S', 'I', ']', "'", 'P', '3', '&', '"',
       'E', 'G', 'Z', 'w', '_', ',', 'r', '0', 'X', 'W', '7', 'H', '2',
       'f', 'l', 'y', '>', '^', 'a', 'h', '\\', '*', '#', ';', '$', 'Q',
       'b', 'i', '1', 'C', '8', '[', 'R', 's', 'J', 'K', 'L', 'q', 'e',
       'x', '?', '4', 'o', 'c', '9', '@', 'v', 'm', '=', 'p', ':', 'M',
       'A', 't', '+', '5', '!', '<', 'u', '`', '6', 'D', 'U', 'T', 'B',
       '-', '.', 'z'], dtype=object)

In [17]:
# Create a quick 'Summary' of flags by month to see the distribution
summary = df_long.groupby(['MONTH', 'QC_FLAG']).size().unstack(fill_value=0)
print(summary)

QC_FLAG     Q      X
MONTH               
1        5412  18757
2        5503  19186
3        5422  18767
4        5592  18599
5        5736  18533
6        5825  18572
7        6040  18524
8        6108  18544
9        5843  18471
10       5596  18415
11       5456  18262
12       5337  17876


In [18]:
df_long.columns

Index(['ID', 'YEAR', 'LATITUDE', 'LONGITUDE', 'NAME', 'MONTH', 'ELEMENT',
       'ELEVATION', 'TEMPERATURE', 'QC_FLAG', 'DM', 'DS'],
      dtype='object')

In [19]:
df_long['DM'].unique()

array([nan])

In [20]:
df_long.drop(columns='DM', inplace=True)

In [21]:
df_long.columns

Index(['ID', 'YEAR', 'LATITUDE', 'LONGITUDE', 'NAME', 'MONTH', 'ELEMENT',
       'ELEVATION', 'TEMPERATURE', 'QC_FLAG', 'DS'],
      dtype='object')

In [28]:
df_long.loc[(df_long['YEAR']==2026) & (df_long['MONTH']==7)]

,ID,YEAR,LATITUDE,LONGITUDE,NAME,MONTH,ELEMENT,ELEVATION,TEMPERATURE,QC_FLAG,DS
2166,AEM00041217,2026,24.4330,54.6510,ABU_DHABI_INTL,7,TAVG,26.8,NaN,None,None
3726,AG000060390,2026,36.7167,3.2500,ALGER_DAR_EL_BEIDA,7,TAVG,24.0,NaN,None,None
4170,AG000060590,2026,30.5667,2.8667,EL_GOLEA,7,TAVG,397.0,NaN,None,None
4614,AG000060611,2026,28.0500,9.6331,IN_AMENAS,7,TAVG,561.0,NaN,None,None
5058,AG000060680,2026,22.8000,5.4331,TAMANRASSET,7,TAVG,1362.0,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...
6710274,VQC00671740,2026,17.7469,-64.7014,CHRISTIANSTED_FT,7,TAVG,9.1,NaN,None,None
6710778,VQW00011624,2026,17.7028,-64.8056,CHRISTIANSTED_AP,7,TAVG,18.6,NaN,None,None
6711210,VQW00011640,2026,18.3331,-64.9667,CHARLOTTE_AMALIE_AP,7,TAVG,6.1,NaN,None,None
6715026,WF000917530,2026,-13.2330,-176.1670,HIHIFO_ILE_WALLIS,7,TAVG,27.0,NaN,None,None


In [29]:
## clean future date
df_long['datetime'] = pd.to_datetime(
    df_long[['YEAR', 'MONTH']].assign(DAY=1)
) 

In [31]:
# clean future date
current_date = datetime.datetime(2026, 3, 31)

# Filter out anything after the current month
df_long = df_long[df_long['datetime'] <= current_date]

In [38]:
df_long.loc[(df_long['YEAR']==2026) & (df_long['MONTH']==2)]

,ID,YEAR,LATITUDE,LONGITUDE,NAME,MONTH,ELEMENT,ELEVATION,TEMPERATURE,QC_FLAG,DS,datetime
2161,AEM00041217,2026,24.4330,54.6510,ABU_DHABI_INTL,2,TAVG,26.8,NaN,None,None,2026-02-01
3721,AG000060390,2026,36.7167,3.2500,ALGER_DAR_EL_BEIDA,2,TAVG,24.0,0.1445,None,P,2026-02-01
4165,AG000060590,2026,30.5667,2.8667,EL_GOLEA,2,TAVG,397.0,0.1605,None,P,2026-02-01
4609,AG000060611,2026,28.0500,9.6331,IN_AMENAS,2,TAVG,561.0,0.1585,None,P,2026-02-01
5053,AG000060680,2026,22.8000,5.4331,TAMANRASSET,2,TAVG,1362.0,0.1700,None,P,2026-02-01
...,...,...,...,...,...,...,...,...,...,...,...,...
6710269,VQC00671740,2026,17.7469,-64.7014,CHRISTIANSTED_FT,2,TAVG,9.1,0.2565,None,7,2026-02-01
6710773,VQW00011624,2026,17.7028,-64.8056,CHRISTIANSTED_AP,2,TAVG,18.6,0.2553,None,D,2026-02-01
6711205,VQW00011640,2026,18.3331,-64.9667,CHARLOTTE_AMALIE_AP,2,TAVG,6.1,0.2699,None,D,2026-02-01
6715021,WF000917530,2026,-13.2330,-176.1670,HIHIFO_ILE_WALLIS,2,TAVG,27.0,0.2785,None,P,2026-02-01


In [34]:
len(df_long['ID'].unique())

22142

In [35]:
len(df_long.loc[(df_long['YEAR']==2026) & (df_long['MONTH']==2)]['ID'].unique())

10098

find the most consistent stations since 2000.

In [39]:
# 1. Define your time range
start_year = 2000
end_year = 2026
end_month = 2 # February (since March is likely incomplete)

# 2. Calculate the total expected months
# (2025 - 2000 + 1) * 12 months + 2 months of 2026
total_expected_months = ((2025 - 2000 + 1) * 12) + 2

# 3. Filter for valid data only
# We exclude NaN temperatures because a 'slot' existing doesn't mean data does
df_valid = df_long[
    (df_long['YEAR'] >= start_year) & 
    (df_long['YEAR'] <= end_year) & 
    (df_long['TEMPERATURE'].notna())
]

# We must also exclude the future months of 2026 we discussed
df_valid = df_valid[~((df_valid['YEAR'] == 2026) & (df_valid['MONTH'] > end_month))]

# 4. Count reporting months per station
station_counts = df_valid.groupby('ID').size()

# 5. Identify the "Perfect" stations
consistent_station_ids = station_counts[station_counts == total_expected_months].index

# 6. Create your final "Gold Standard" dataframe
df_consistent = df_long[df_long['ID'].isin(consistent_station_ids)].copy()

print(f"Found {len(consistent_station_ids)} stations with 100% reporting consistency since 2000.")

Found 1054 stations with 100% reporting consistency since 2000.


In [42]:
df_consistent['ID'].unique().shape

(1054,)

In [44]:
most_consistent_stations = df_consistent['ID'].unique()

In [45]:
most_consistent_stations

array(['AGE00147718', 'AGM00060620', 'ARM00087046', ..., 'UZM00038696',
       'UZM00038927', 'WF000917530'], shape=(1054,), dtype=object)

In [49]:
folder = 'data'
pd.Series(most_consistent_stations,name='id_stations').to_csv(os.path.join(folder,
                                                                            'most_consistent_stations-2000-2026.csv'))

In [47]:
folder

NameError: name 'folder' is not defined